# Subtitle Generator and Summarizer

## How to Run This Notebook

### Requirements
Install all dependencies first:
pip install -r requirements.txt
sudo apt install ffmpeg

### Input
Place 3 lecture videos (10–15 min, .mp4) in the /videos folder.
Recommended test videos (freely available):
- MIT 18.06 Linear Algebra Lecture 1:
  https://www.youtube.com/watch?v=ZK3O402wf1c
- Khan Academy Intro to Vectors:
  https://www.youtube.com/watch?v=br7tS1t2SFE
- freeCodeCamp Python Basics:
  https://www.youtube.com/watch?v=rfscVS0vtbw

Download using: yt-dlp -f mp4 -o "videos/lectureN.mp4" [URL]

### Run Order
Execute cells top to bottom. Each section depends on the previous.
Expected total runtime: 15–30 minutes depending on hardware.

### Outputs
- /subtitles  → lecture1.srt, lecture2.srt, lecture3.srt
- /summaries  → lecture1_summary.txt, lecture2_summary.txt, 
                 lecture3_summary.txt
- /reports    → wer_results.txt, rouge_results.txt, 
                 final_report.md

# End-to-End Subtitle Generator and Summarizer Pipeline

This notebook demonstrates the complete progression from a raw educational `.mp4` video to generated `.srt` subtitle files and concise `.txt` summaries using OpenAI Whisper and Hugging Face BART.

**File Structure Expectations:**
* `videos/lecture1.mp4` must exist before running Audio Extraction.
* Outputs populate dynamically into `audio/`, `transcripts/`, `subtitles/`, `summaries/`, `reports/`.

*Helper functions `chunk_transcript` and `convert_to_srt` are imported from `utils.py` — no logic is duplicated here.*

## Helper Functions

In [1]:
try:
    from utils import convert_to_srt, chunk_transcript
except ImportError:
    # Inline definitions follow — notebook is self-contained
    def format_timestamp(seconds: float) -> str:
        if seconds < 0:
            seconds = 0.0
    
        hours   = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs    = int(seconds % 60)
        millis  = int(round((seconds - int(seconds)) * 1000))
    
        if millis >= 1000:
            millis = 0
            secs += 1
            if secs >= 60:
                secs = 0
                minutes += 1
                if minutes >= 60:
                    minutes = 0
                    hours += 1
    
        return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"
    
    def convert_to_srt(segments: list[dict], output_path) -> 'Path':
        from pathlib import Path
        output_path = Path(output_path)
    
        if not segments:
            raise ValueError("Segment list is empty — cannot generate SRT file.")
    
        required_keys = {"start", "end", "text"}
        srt_blocks = []
    
        for idx, seg in enumerate(segments, start=1):
            missing = required_keys - seg.keys()
            if missing:
                raise ValueError(
                    f"Segment {idx} is missing required key(s): {missing}"
                )
    
            start_ts = format_timestamp(seg["start"])
            end_ts   = format_timestamp(seg["end"])
            text     = seg["text"].strip()
    
            srt_blocks.append(f"{idx}\n{start_ts} --> {end_ts}\n{text}\n")
    
        srt_content = "\n".join(srt_blocks) + "\n"
    
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(srt_content, encoding="utf-8")
    
        return output_path.resolve()
    
    def chunk_transcript(text: str, max_words: int = 200) -> list[str]:
        import nltk
        if not text or not text.strip():
            raise ValueError("Empty transcript passed to chunk_transcript")
            
        try:
            nltk.data.find('tokenizers/punkt_tab')
        except LookupError:
            nltk.download('punkt', quiet=True)
            nltk.download('punkt_tab', quiet=True)
    
        sentences = nltk.sent_tokenize(text.strip())
        chunks = []
        current_chunk = []
        current_word_count = 0
    
        for sentence in sentences:
            word_count = len(sentence.split())
            
            if current_word_count + word_count > max_words and current_chunk:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentence]
                current_word_count = word_count
            else:
                current_chunk.append(sentence)
                current_word_count += word_count
    
        if current_chunk:
            chunks.append(" ".join(current_chunk))
    
        return chunks

## Setup PATHS

In [2]:
import os
from pathlib import Path

# ── Project root (works from any working directory) ──
ROOT = Path(__file__).parent if '__file__' in dir() else Path.cwd()

PATHS = {
    "videos":      ROOT / "videos",
    "audio":       ROOT / "audio",
    "transcripts": ROOT / "transcripts",
    "subtitles":   ROOT / "subtitles",
    "summaries":   ROOT / "summaries",
    "reports":     ROOT / "reports",
    "notebooks":   ROOT / "notebooks",
}

# Create all folders if they don't exist
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("✅ All paths configured:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")

✅ All paths configured:
  videos: /home/arth-linux/Desktop/My Code/Subtitle Generator/notebooks/videos
  audio: /home/arth-linux/Desktop/My Code/Subtitle Generator/notebooks/audio
  transcripts: /home/arth-linux/Desktop/My Code/Subtitle Generator/notebooks/transcripts
  subtitles: /home/arth-linux/Desktop/My Code/Subtitle Generator/notebooks/subtitles
  summaries: /home/arth-linux/Desktop/My Code/Subtitle Generator/notebooks/summaries
  reports: /home/arth-linux/Desktop/My Code/Subtitle Generator/notebooks/reports
  notebooks: /home/arth-linux/Desktop/My Code/Subtitle Generator/notebooks/notebooks


## 1. Setup and Imports

In [3]:
# Standard library
import os, re, json, zipfile
from pathlib import Path

# Audio/video
import ffmpeg
from pydub import AudioSegment

# Whisper
import whisper

# Summarization
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

# Evaluation
import jiwer
from rouge_score import rouge_scorer

# Notebook display
from IPython.display import display, Markdown, Audio

# Verify all imports succeeded
print("✅ All libraries imported successfully")
print(f"   Whisper version : {whisper.__version__}")

✅ All libraries imported successfully
   Whisper version : 20250625


In [4]:
# Verify ffmpeg is available on system PATH
import subprocess
result = subprocess.run(['ffmpeg', '-version'], 
                        capture_output=True, text=True)
if result.returncode == 0:
    print("✅ ffmpeg is available")
else:
    print("❌ ffmpeg not found — install with: sudo apt install ffmpeg")

✅ ffmpeg is available


## 2. Audio Extraction

In [5]:
import subprocess
for video_file in PATHS['videos'].glob('*.mp4'):
    try:
        # validate file is not corrupt before processing
        result = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 
             'format=duration', '-of', 
             'default=noprint_wrappers=1:nokey=1',
             str(video_file)],
            capture_output=True, text=True
        )
        if not result.stdout.strip():
            print(f"⚠️  Skipping {video_file.name} — unreadable")
            continue
        duration = float(result.stdout.strip())
        if duration < 60:
            print(f"⚠️  Skipping {video_file.name} — too short")
            continue
        # proceed with extraction
        print(f"Processing {video_file.name} ({duration:.0f}s)...")
        
        aud_path = PATHS["audio"] / f"{video_file.stem}.wav"
        (
            ffmpeg.input(str(video_file))
            .output(str(aud_path), ac=1, ar=16000, format="wav")
            .overwrite_output()
            .run(quiet=True)
        )
        size_mb = aud_path.stat().st_size / 1e6
        print(f"✅ Saved {aud_path.name} ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"❌ Failed on {video_file.name}: {e}")
        continue

## 3. Whisper Transcription (with Segments)

In [6]:
import torch
import time

for audio_file in PATHS['audio'].glob('*.wav'):
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading Whisper 'base' on {device}...")
        model = whisper.load_model("base", device=device)
        print(f"Transcribing {audio_file.name}...  (may take a few minutes)")
        start = time.time()
        result = model.transcribe(str(audio_file))
        elapsed = time.time() - start
        
        txt_path = PATHS["transcripts"] / f"{audio_file.stem}.txt"
        json_path = PATHS["transcripts"] / f"{audio_file.stem}_segments.json"
        
        txt_path.write_text(result["text"].strip(), encoding="utf-8")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump({"segments": result["segments"]}, f, indent=2)
            
        print(f"✅ Done in {elapsed:.1f}s — {len(result['text'].split())} words")
        print(f"Saved: {txt_path.name}, {json_path.name}")
    except Exception as e:
        print(f"❌ Failed on {audio_file.name}: {e}")
        continue

## 4. SRT Subtitle Generation

In [7]:
for json_path in PATHS['transcripts'].glob('*_segments.json'):
    try:
        srt_path = PATHS["subtitles"] / f"{json_path.name.replace('_segments.json', '.srt')}"
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        segments = data.get("segments", [])
        
        convert_to_srt(segments, srt_path)
        blocks = [b for b in srt_path.read_text().split("\n\n") if b.strip()]
        print(f"✅ {srt_path.name} written — {len(blocks)} subtitle blocks")
    except Exception as e:
        print(f"❌ Failed on {json_path.name}: {e}")
        continue

In [8]:
# Display first 20 lines of each .srt file
for v in ['lecture1', 'lecture2', 'lecture3']:
    srt_path = PATHS['subtitles'] / f'{v}.srt'
    if srt_path.exists():
        content = srt_path.read_text()
        preview = '\n'.join(content.split('\n')[:20])
        display(Markdown(f"### {v}.srt preview\n```\n{preview}\n```"))
    else:
        print(f"⚠️ {srt_path.name} not found")

⚠️ lecture1.srt not found
⚠️ lecture2.srt not found
⚠️ lecture3.srt not found


## 5. Transcript Chunking

In [9]:
chunked_data = {}
for txt_path in PATHS['transcripts'].glob('*.txt'):
    # Ignore the _segments.json if glob matched incorrectly, but it's .txt
    if txt_path.name.endswith('_segments.txt'): continue
    
    try:
        raw_text = txt_path.read_text(encoding="utf-8")
        chunks = chunk_transcript(raw_text, max_words=200)
        chunked_data[txt_path.stem] = chunks
        print(f"✅ {txt_path.name}: {len(chunks)} chunks created")
        if chunks:
            print(f"   Chunk 1 preview ({len(chunks[0].split())} words): {chunks[0][:120]}...")
    except Exception as e:
        print(f"❌ Failed on {txt_path.name}: {e}")
        continue

## 6. Summarization with BART

In [10]:
# Model loading with fallback
try:
    model_name = "facebook/bart-large-cnn"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    print(f"✅ Loaded {model_name}")
except Exception:
    model_name = "google/flan-t5-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    print(f"⚠️  Fell back to {model_name}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for stem, chunks in chunked_data.items():
    try:
        summary_path = PATHS["summaries"] / f"{stem}_summary.txt"
        intermediate = []
        for idx, chunk in enumerate(chunks, 1):
            inputs = tokenizer(chunk, max_length=1024, return_tensors="pt", truncation=True).to(device)
            ids = model.generate(inputs["input_ids"], max_length=60, min_length=15, num_beams=4, early_stopping=True)
            intermediate.append(tokenizer.decode(ids[0], skip_special_tokens=True))

        combined = " ".join(intermediate)
        
        # Enforce ≤100-word hard limit
        if len(combined.split()) > 100:
            inputs = tokenizer(combined, max_length=1024, return_tensors="pt", truncation=True).to(device)
            ids = model.generate(inputs["input_ids"], max_length=95, min_length=30, num_beams=4, early_stopping=True)
            combined = tokenizer.decode(ids[0], skip_special_tokens=True)

        summary_path.write_text(combined.strip(), encoding="utf-8")
        print(f"✅ Summary saved for {stem} ({len(combined.split())} words)")
    except Exception as e:
        print(f"❌ Failed on {stem}: {e}")
        continue

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

✅ Loaded facebook/bart-large-cnn


In [11]:
# Display all 3 summaries with word and sentence count
for v in ['lecture1', 'lecture2', 'lecture3']:
    summary_file = PATHS['summaries'] / f'{v}_summary.txt'
    if summary_file.exists():
        s = summary_file.read_text().strip()
        words = len(s.split())
        sents = len(re.findall(r'[.!?]+', s))
        display(Markdown(
            f"### {v} Summary\n{s}\n\n"
            f"*{words} words · {sents} sentences*"
        ))
    else:
        print(f"⚠️ {summary_file.name} not found")

⚠️ lecture1_summary.txt not found
⚠️ lecture2_summary.txt not found
⚠️ lecture3_summary.txt not found


## 7. WER Evaluation

In [12]:
with open(PATHS['reports'] / 'wer_results.txt', 'w') as f_out:
    for txt_path in PATHS['transcripts'].glob('*.txt'):
        try:
            stem = txt_path.stem
            ref_path = ROOT / "references" / f"{stem}.txt"
            if ref_path.exists():
                ref = " ".join(ref_path.read_text(encoding="utf-8").split()).lower()
                hyp = " ".join(txt_path.read_text(encoding="utf-8").split()).lower()
                wer = jiwer.wer(ref, hyp)
                res = f"{stem} WER: {wer:.2%}\n"
                print(res.strip())
                print("✅ PASS" if wer < 0.30 else "⚠️  FAIL — consider Whisper 'medium' or 'large'")
                f_out.write(res)
            else:
                print(f"⚠️  Reference transcript not found for {stem}.")
        except Exception as e:
            print(f"❌ Failed evaluating WER for {stem}: {e}")
            continue

## 8. ROUGE Evaluation

In [13]:
with open(PATHS['reports'] / 'rouge_results.txt', 'w') as f_out:
    for summary_path in PATHS['summaries'].glob('*_summary.txt'):
        try:
            stem = summary_path.stem.replace('_summary', '')
            ref_path = ROOT / "reference_summaries" / f"{stem}_summary.txt"
            if ref_path.exists():
                ref_sum = ref_path.read_text(encoding="utf-8").strip()
                hyp_sum = summary_path.read_text(encoding="utf-8").strip()
                scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
                scores = scorer.score(ref_sum, hyp_sum)
                res = f"{stem} - ROUGE-1: {scores['rouge1'].fmeasure:.4f}, ROUGE-2: {scores['rouge2'].fmeasure:.4f}, ROUGE-L: {scores['rougeL'].fmeasure:.4f}\n"
                print(res.strip())
                f_out.write(res)
            else:
                print(f"⚠️  Reference summary not found for {stem}.")
        except Exception as e:
            print(f"❌ Failed evaluating ROUGE for {stem}: {e}")
            continue

In [14]:
# Display WER and ROUGE scores as a results table
wer_path = PATHS['reports'] / 'wer_results.txt'
rouge_path = PATHS['reports'] / 'rouge_results.txt'

if wer_path.exists():
    wer_text = wer_path.read_text()
    display(Markdown("### WER Results\n```\n" + wer_text + "\n```"))
    
if rouge_path.exists():
    rouge_text = rouge_path.read_text()
    display(Markdown("### ROUGE Results\n```\n" + rouge_text + "\n```"))

### WER Results
```

```

### ROUGE Results
```

```